# PS LiDAR - Laboratorio de Desarrollo

**Ladrillos disponibles:**
- Brick 1: Carga de Datos
- Brick 2: Recorte Circular (coordenadas manuales)
- Brick 3: Detección de Normalización
- Brick 4: Filtrado de Suelo
- Brick 5: Normalización de Altura
- Brick 5.5: Exportar Checkpoints
- Brick 6: Visualización 3D

In [1]:
import os
import sys
import time
from pathlib import Path

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.core import (
    PointCloudLoader,
    detect_normalization,
    classify_ground,
    clip_circular_plot,
    normalize_heights,
    export_point_cloud,
)

print("✓ Módulos importados")

✓ Módulos importados


---
## 1. Cargar Archivo (Brick 1)

In [2]:
FILE_PATH = "C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/HQP080_01_raw.laz"

loader = PointCloudLoader(FILE_PATH)
loader.load()

meta = loader.get_metadata()
print(f"Archivo: {meta['filename']}")
print(f"Puntos: {meta['point_count']:,}")
print(f"Tamaño: {meta['file_size_mb']} MB")

Archivo: HQP080_01_raw.laz
Puntos: 23,934,061
Tamaño: 258.2 MB


In [3]:
# Cargar XYZ y campos escalares disponibles
xyz_full = loader.get_xyz()

scalar_fields = {}
for field in ['intensity', 'return_number', 'number_of_returns', 'classification']:
    try:
        scalar_fields[field] = loader.get_attribute(field)
        print(f"✓ {field}: {len(scalar_fields[field]):,} valores")
    except:
        print(f"⚠ {field}: no disponible")

print(f"\nMemoria XYZ: {xyz_full.nbytes / (1024**2):.1f} MB")

✓ intensity: 23,934,061 valores
✓ return_number: 23,934,061 valores
✓ number_of_returns: 23,934,061 valores
✓ classification: 23,934,061 valores

Memoria XYZ: 547.8 MB


---
## 2. Recorte Circular (Brick 2)

**Instrucciones:**
1. Abrir el archivo original en CloudCompare
2. Usar herramienta "Point Picking" para ubicar el centro del mat
3. Copiar las coordenadas Xg, Yg mostradas
4. Pegar los valores en `CENTER_X` y `CENTER_Y` abajo

In [4]:
# ═══════════════════════════════════════════════════════════════
# PARÁMETROS DEL USUARIO - Modificar según el plot
# ═══════════════════════════════════════════════════════════════

# Coordenadas del centro (obtenidas de CloudCompare Point Picking)
# Xg
CENTER_X = -0.809949
# Yg
CENTER_Y = -0.745654

# Radio del plot en metros
PLOT_RADIUS = 16.0

# ═══════════════════════════════════════════════════════════════

print(f"Centro: ({CENTER_X:.6f}, {CENTER_Y:.6f})")
print(f"Radio: {PLOT_RADIUS}m")

Centro: (-0.809949, -0.745654)
Radio: 16.0m


In [5]:
# Ejecutar recorte circular
t0 = time.perf_counter()
clip_result = clip_circular_plot(xyz_full, CENTER_X, CENTER_Y, PLOT_RADIUS)
elapsed = time.perf_counter() - t0

plot_indices = clip_result.indices

print(f"✓ Recorte en {elapsed*1000:.0f}ms")
print(f"Puntos originales: {len(xyz_full):,}")
print(f"Puntos en plot: {clip_result.n_points:,} ({clip_result.n_points/len(xyz_full):.1%})")

✓ Recorte en 936ms
Puntos originales: 23,934,061
Puntos en plot: 15,511,622 (64.8%)


In [6]:
# Aplicar recorte a XYZ y campos escalares
xyz = xyz_full[plot_indices]

plot_scalars = {}
for field, values in scalar_fields.items():
    plot_scalars[field] = values[plot_indices]

print(f"Plot XYZ: {xyz.shape}")
print(f"Campos escalares: {list(plot_scalars.keys())}")

# Liberar memoria
del xyz_full, scalar_fields
import gc; gc.collect()
print("✓ Memoria liberada")

Plot XYZ: (15511622, 3)
Campos escalares: ['intensity', 'return_number', 'number_of_returns', 'classification']
✓ Memoria liberada


---
## 3. Análisis de Normalización (Brick 3)

In [7]:
analysis = detect_normalization(xyz)
print(f"Estatus: {analysis.status.value.upper()}")
print(f"¿Normalizada?: {analysis.is_normalized}")
print(f"Rango Z: {analysis.z_min:.2f}m a {analysis.z_max:.2f}m")

Estatus: NOT_NORMALIZED
¿Normalizada?: False
Rango Z: -2.28m a 17.58m


---
## 4. Filtrado de Suelo (Brick 4)

In [8]:
print("Ejecutando CSF...")
t0 = time.perf_counter()

ground_result = classify_ground(
    xyz,
    cloth_resolution=1.0,
    rigidness=1,
    class_threshold=0.5,
    slope_smooth=True,
)

print(f"✓ Completado en {time.perf_counter() - t0:.2f}s")
print(f"Suelo: {ground_result.n_ground:,} ({ground_result.ground_ratio:.1%})")
print(f"Vegetación: {ground_result.n_off_ground:,}")

Ejecutando CSF...
✓ Completado en 7.01s
Suelo: 3,891,105 (25.1%)
Vegetación: 11,620,517


In [9]:
# Separar suelo y vegetación
ground_xyz = xyz[ground_result.ground_indices]
vegetation_xyz = xyz[ground_result.off_ground_indices]

ground_scalars = {k: v[ground_result.ground_indices] for k, v in plot_scalars.items()}
vegetation_scalars = {k: v[ground_result.off_ground_indices] for k, v in plot_scalars.items()}

print(f"Suelo: {len(ground_xyz):,} puntos")
print(f"Vegetación: {len(vegetation_xyz):,} puntos")

Suelo: 3,891,105 puntos
Vegetación: 11,620,517 puntos


---
## 5. Normalización de Altura (Brick 5)

In [10]:
print("Normalizando alturas...")
t0 = time.perf_counter()

veg_norm_result = normalize_heights(vegetation_xyz, ground_xyz, resolution=0.5)
veg_normalized = veg_norm_result.xyz_normalized

ground_norm_result = normalize_heights(ground_xyz, ground_xyz, resolution=0.5)
ground_normalized = ground_norm_result.xyz_normalized

print(f"✓ Completado en {(time.perf_counter() - t0)*1000:.0f}ms")
print(f"")
print(f"Vegetación: Z = {veg_normalized[:, 2].min():.2f}m a {veg_normalized[:, 2].max():.2f}m")
print(f"Suelo: Z = {ground_normalized[:, 2].min():.2f}m a {ground_normalized[:, 2].max():.2f}m")

Normalizando alturas...
✓ Completado en 2821ms

Vegetación: Z = -0.93m a 19.58m
Suelo: Z = -0.76m a 0.79m


---
## 5.5 Exportar Checkpoints

In [ ]:
# Directorio de salida
OUTPUT_DIR = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw")

# Exportar vegetación
veg_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/vegetation_normalized_3.laz")

export_point_cloud(
    veg_file,
    veg_normalized,
    intensity=vegetation_scalars.get('intensity'),
    return_number=vegetation_scalars.get('return_number'),
    number_of_returns=vegetation_scalars.get('number_of_returns'),
    classification=vegetation_scalars.get('classification'),
)
print(f"✓ Vegetación: {veg_file.name} ({veg_file.stat().st_size / (1024**2):.1f} MB)")

# Exportar suelo
ground_file = Path("C:/Users/geoal/Documents/Work/LidarProcessing/LiDAR Cline/Data/raw/ground_normalized.laz")
export_point_cloud(
    ground_file,
    ground_normalized,
    intensity=ground_scalars.get('intensity'),
    return_number=ground_scalars.get('return_number'),
    number_of_returns=ground_scalars.get('number_of_returns'),
    classification=ground_scalars.get('classification'),
)
print(f"✓ Suelo: {ground_file.name} ({ground_file.stat().st_size / (1024**2):.1f} MB)")

---
## 6. Visualización 3D (Brick 6)

In [ ]:
import open3d as o3d
import numpy as np

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(veg_normalized)

# Colorear por altura
z = veg_normalized[:, 2]
z_scaled = (z - z.min()) / (z.max() - z.min() + 1e-6)
colors = np.zeros((len(z_scaled), 3))
colors[:, 0] = z_scaled
colors[:, 1] = 1 - np.abs(2 * z_scaled - 1)
colors[:, 2] = 1 - z_scaled
pcd.colors = o3d.utility.Vector3dVector(colors)

print(f"Nube: {len(pcd.points):,} puntos")

In [ ]:
o3d.visualization.draw_geometries([pcd], window_name="Vegetación Normalizada", width=1280, height=720)

---
## 7. Próximos Pasos

- **Brick 7:** Segmentación de Árboles
- **Brick 8:** Métricas por árbol (DBH, altura)